In [ ]:
from google.colab import drive
drive.mount('/content/drive')




Mounted at /content/drive


In [ ]:
DIR="/content/drive/MyDrive/NLP"
MODEL_DIR= "/content/drive/MyDrive/NLP/ngram_model"

In [ ]:
import random
import pickle
from collections import Counter
import numpy as np
from collections import defaultdict
import math
import json
import numpy as np
from itertools import product
import os


In [ ]:

def load_model(filename):
    with open(filename, "rb") as f:
        model = pickle.load(f)
    return model

In [ ]:
def load_sentences(filename):
    sentences = []
    with open(filename, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                if line.startswith("[") and line.endswith("]"):
                    sent = line[1:-1].split(", ")
                    sent = [w.strip("'\"") for w in sent]
                    sentences.append(sent)
                else:
                    sentences.append(line.split())
    return sentences



In [ ]:

bigram_model=load_model(f'{MODEL_DIR}/final_2gram_counts.pkl')
uni_model=load_model(f'{MODEL_DIR}//final_1gram_counts.pkl')
tri_model=load_model(f'{MODEL_DIR}/final_3gram_counts.pkl')
quad_model=load_model(f'{MODEL_DIR}/final_4gram_counts.pkl')



In [ ]:
test_sentences = load_sentences(f"{MODEL_DIR}/test_sentences.csv")
print(f"Loaded {len(test_sentences)} test sentences.")
validation_sentences=load_sentences(f"{MODEL_DIR}/val_sentences.csv")
print(f"Loaded {len(validation_sentences)} val sentences.")
news_sentences=load_sentences(f"{MODEL_DIR}/1000_news_sentences.txt")
print(f"Loaded {len(news_sentences)} news sentences.")


Loaded 1001 test sentences.
Loaded 1001 val sentences.
Loaded 1000 news sentences.


In [ ]:
%whos

Variable         Type        Data/Info
--------------------------------------
Counter          type        <class 'collections.Counter'>
DIR              str         /content/drive/MyDrive/NLP
MODEL_DIR        str         /content/drive/MyDrive/NLP/ngram_model
bigram_model     dict        n=5143767
defaultdict      type        <class 'collections.defaultdict'>
drive            module      <module 'google.colab.dri<...>s/google/colab/drive.py'>
load_model       function    <function load_model at 0x792a44e1e520>
load_sentences   function    <function load_sentences at 0x792a0f5714e0>
math             module      <module 'math' (built-in)>
np               module      <module 'numpy' from '/us<...>kages/numpy/__init__.py'>
pickle           module      <module 'pickle' from '/u<...>ib/python3.12/pickle.py'>
quad_model       dict        n=12034694
random           module      <module 'random' from '/u<...>ib/python3.12/random.py'>
test_sentences   list        n=1001
tri_model        dict  

# Interpolitation

## fast

In [ ]:

def deleted_interpolation_fast(heldout_sentences, uni_model, bi_model, tri_model, quad_model):
    uni_counts = uni_model
    bi_counts = bi_model
    tri_counts = tri_model
    quad_counts = quad_model

    total_unigrams = sum(uni_counts.values())
    lambda_counts = np.zeros(4, dtype=np.float64)
    eps = 1e-12

    for sent in heldout_sentences:
        tokens = sent
        padded = ["<s>", "<s>", "<s>"] + tokens + ["</s>"] * 3

        for i in range(3, len(padded)):
            w4 = padded[i]
            w3 = padded[i-1]
            w2 = padded[i-2]
            w1 = padded[i-3]

            # counts
            quad_count = quad_counts.get((w1, w2, w3, w4), 0)
            tri_count  = tri_counts.get((w1, w2, w3), 0)
            bi_count   = bi_counts.get((w2, w3), 0)
            uni_count  = uni_counts.get((w3,), 0)

            # conditional probabilities
            p4 = quad_count / (tri_count + eps)
            p3 = tri_counts.get((w2, w3, w4), 0) / (bi_count + eps)
            p2 = bi_counts.get((w3, w4), 0) / (uni_count + eps)
            p1 = uni_counts.get((w4,), 0) / total_unigrams

            ps = [p1, p2, p3, p4]
            max_idx = np.argmax(ps)
            lambda_counts[max_idx] += 1

    lambdas = lambda_counts / np.sum(lambda_counts)
    return lambdas


## prob

In [ ]:

def quadrigram_prob(sentence, uni_model, bi_model, tri_model, quad_model, lambdas):

    lambda1, lambda2, lambda3, lambda4 = lambdas
    log_prob = 0.0
    tokens = sentence
    padded = ["<s>"]*3 + tokens + ["</s>"]

    for i in range(3, len(padded)):
        w4 = padded[i]
        w3 = padded[i-1]
        w2 = padded[i-2]
        w1 = padded[i-3]


        p4 = quad_model.get((w1,w2,w3,w4),0) / max(tri_model.get((w1,w2,w3),1),1)
        p3 = tri_model.get((w2,w3,w4),0) / max(bi_model.get((w2,w3),1),1)
        p2 = bi_model.get((w3,w4),0) / max(uni_model.get((w3,),1),1)
        p1 = uni_model.get((w4,),0) / max(sum(uni_model.values()),1)

        p = lambda4*p4 + lambda3*p3 + lambda2*p2 + lambda1*p1
        log_prob += math.log(p + 1e-12)

    return log_prob



## evaluate lemda

In [ ]:

def evaluate_lambdas(heldout_sentences, uni_model, bi_model, tri_model, quad_model, lambdas):
    total_unigrams = sum(uni_model.values())
    eps = 1e-12
    error = 0.0
    count = 0

    for sent in heldout_sentences:
        tokens = sent
        padded = ["<s>", "<s>", "<s>"] + tokens + ["</s>"]*3

        for i in range(3, len(padded)):
            w4 = padded[i]; w3 = padded[i-1]; w2 = padded[i-2]; w1 = padded[i-3]

            # counts
            quad_count = quad_model.get((w1, w2, w3, w4), 0)
            tri_count  = tri_model.get((w1, w2, w3), 0)
            bi_count   = bi_model.get((w2, w3), 0)
            uni_count  = uni_model.get((w3,), 0)

            # probs
            p4 = quad_count / (tri_count + eps)
            p3 = tri_model.get((w2, w3, w4), 0) / (bi_count + eps)
            p2 = bi_model.get((w3, w4), 0) / (uni_count + eps)
            p1 = uni_model.get((w4,), 0) / total_unigrams

            eps = 1e-12  # already used

            # Add smoothing to avoid zeros dominating
            p4 = max(p4, eps)
            p3 = max(p3, eps)
            p2 = max(p2, eps)
            p1 = max(p1, eps)


            # interpolated prob
            p_interp = lambdas[0]*p1 + lambdas[1]*p2 + lambdas[2]*p3 + lambdas[3]*p4

            # error (MSE)
            # error += (p_interp - p4)**2
            error += -math.log(p_interp)
            count += 1

    return error / max(1, count)


## grid

In [ ]:
def deleted_interpolation_gridsearch(heldout_sentences, uni_model, bi_model, tri_model, quad_model):
    best_lambdas = None
    best_error = float("inf")

    # step size for lambda search
    step = 0.2  # try 0.1 for finer grid

    for l1, l2, l3, l4 in product(np.arange(0, 1.01, step), repeat=4):
        if abs(l1 + l2 + l3 + l4 - 1.0) > 1e-6:
            continue  # must sum to 1

        lambdas = [l1, l2, l3, l4]
        error = evaluate_lambdas(heldout_sentences, uni_model, bi_model, tri_model, quad_model, lambdas)

        if error < best_error:
            best_error = error
            best_lambdas = lambdas

    return best_lambdas, best_error

In [ ]:



lambdas = deleted_interpolation_fast(validation_sentences, uni_model, bigram_model, tri_model, quad_model)
print("Best λ values:", lambdas)
lambda_file = f"{DIR}/Assignment 5/best_lambdas.json"

# os.makedirs(lambda_file, exist_ok=True)


with open(lambda_file, "w",encoding='utf-8') as f:
    json.dump({"lambdas": lambdas.tolist()}, f, indent=4)










Best λ values: [0.43938905 0.09855031 0.18828057 0.27378007]


In [ ]:
for sent in test_sentences[:5]:
    lp = quadrigram_prob(sent, uni_model, bigram_model, tri_model, quad_model, lambdas)
    print(f"Sentence: {sent}  Log-prob: {lp}")

Sentence: ['sentence']  Log-prob: -14.357439561212212
Sentence: ['Due', 'to', 'such', 'vairous', 'in', 'yahoo', 'mail', 'my', 'all', 'address', 'book', 'is', 'also', 'lost', 'and', 'hence', 'not', 'able', 'to', 'send', 'you', 'individual', 'mails', '.']  Log-prob: -28.461799580653356
Sentence: ['અલબત્ત', 'ગીતા', 'અને', 'ભાગવત', 'મારાં', 'શાળાના', 'દિવસો', 'દરમિયાન', 'વાંચ્યા', 'છે', '.']  Log-prob: -18.763487933912312
Sentence: ['"આથી', 'જ', 'ગુજરાતીલેક્સિકોન', 'મોબાઇલ', 'ટૅક્નૉલૉજીના', 'યુગમાં', 'નવીન', 'ટૅક્નૉલૉજી', 'સાથે', 'તાલથી', 'તાલ', 'મેળવીને', 'તેની', 'વિવિધ', 'પાંચ', 'મોબાઇલ', 'એપ્લિકેશન', 'રજૂ', 'કરી', 'છે', '.', 'n', 'આ', 'એપ્લિકેશન', 'એન્ડ્રોઇડ', ',', 'એપલ', 'આઇઓએસ', 'અને', 'બ્લેકબેરી', 'ધરાવતા', 'મોટાભાગના', 'સ્માર્ટફોન', 'અને', 'ટેબલેટમાં', 'પણ', 'રમી', 'શકાશે', '.', 'n', 'ગુજરાતીલેક્સિકોન', 'દ્વારા', 'રજૂ', 'થતી', 'પાંચ', 'ઍપ્લિકેશનની', 'માહિતી', 'નીચે', 'પ્રમાણે', 'છે', ':', 'n1', '."']  Log-prob: -122.18055141309668
Sentence: ['HappyBhaiDooj', 'BhaiDooj', 'BhaiDooj202

In [ ]:
best_lambdas_grid, best_error = deleted_interpolation_gridsearch(
    heldout_sentences=validation_sentences,
    uni_model=uni_model,
    bi_model=bigram_model,
    tri_model=tri_model,
    quad_model=quad_model
)

print("Grid search λ values:", best_lambdas_grid)
print("Grid search MSE:", best_error)


Grid search λ values: [np.float64(0.2), np.float64(0.2), np.float64(0.2), np.float64(0.4)]
Grid search MSE: 7.786444490102596
 Grid search λ values saved to: /content/drive/MyDrive/NLP/Assignment 5/best_lambdas_gridsearch.json


In [ ]:
grid_file = f"{DIR}/Assignment 5/best_lambdas_gridsearch.json"
with open(grid_file, "w") as f:
    json.dump({"lambdas": best_lambdas_grid, "MSE": best_error}, f, indent=4)

# compare

In [ ]:
print("Deleted Interpolation Fast λ:", lambdas)
print("Grid Search λ:", best_lambdas_grid)


Deleted Interpolation Fast λ: [0.43938905 0.09855031 0.18828057 0.27378007]
Grid Search λ: [np.float64(0.2), np.float64(0.2), np.float64(0.2), np.float64(0.4)]


## compare test sentences

In [ ]:
for sent in test_sentences:
    lp_fast = quadrigram_prob(sent, uni_model, bigram_model, tri_model, quad_model, lambdas)
    lp_grid = quadrigram_prob(sent, uni_model, bigram_model, tri_model, quad_model, best_lambdas_grid)
    print(f"Sentence: {' '.join(sent)}")
    print(f" Log-prob (Fast λ): {lp_fast}")
    print(f" Log-prob (Grid λ): {lp_grid}\n")


Sentence: sentence
 Log-prob (Fast λ): -14.357439561212212
 Log-prob (Grid λ): -15.931575293911266

Sentence: Due to such vairous in yahoo mail my all address book is also lost and hence not able to send you individual mails .
 Log-prob (Fast λ): -28.461799580653356
 Log-prob (Grid λ): -20.732385921390414

Sentence: અલબત્ત ગીતા અને ભાગવત મારાં શાળાના દિવસો દરમિયાન વાંચ્યા છે .
 Log-prob (Fast λ): -18.763487933912312
 Log-prob (Grid λ): -14.860115692774242

Sentence: "આથી જ ગુજરાતીલેક્સિકોન મોબાઇલ ટૅક્નૉલૉજીના યુગમાં નવીન ટૅક્નૉલૉજી સાથે તાલથી તાલ મેળવીને તેની વિવિધ પાંચ મોબાઇલ એપ્લિકેશન રજૂ કરી છે . n આ એપ્લિકેશન એન્ડ્રોઇડ , એપલ આઇઓએસ અને બ્લેકબેરી ધરાવતા મોટાભાગના સ્માર્ટફોન અને ટેબલેટમાં પણ રમી શકાશે . n ગુજરાતીલેક્સિકોન દ્વારા રજૂ થતી પાંચ ઍપ્લિકેશનની માહિતી નીચે પ્રમાણે છે : n1 ."
 Log-prob (Fast λ): -122.18055141309668
 Log-prob (Grid λ): -108.65222043373196

Sentence: HappyBhaiDooj BhaiDooj BhaiDooj2022 Siblinghood IndianFestivals Celebration HappyDiwali FestiveSeason Book BookLo

## compare news sentences




In [ ]:
for sent in news_sentences:
    lp_fast = quadrigram_prob(sent, uni_model, bigram_model, tri_model, quad_model, lambdas)
    lp_grid = quadrigram_prob(sent, uni_model, bigram_model, tri_model, quad_model, best_lambdas_grid)
    print(f"Sentence: {' '.join(sent)}")
    print(f" Log-prob (Fast λ): {lp_fast}")
    print(f" Log-prob (Grid λ): {lp_grid}\n")

Sentence: First Bangladeshi to hit three sixes in an ODI, Shorna Akter also scripted history with the fastest fifty in just 34 balls!#CricketTwitter#CWC25#BANvSApic.twitter.com/Ezq629Y2Ot — Female Cricket (@imfemalecricket)October 13, 2025 આ પહેલીવાર નથી જ્યારે શોર્ના અખ્તરે પોતાના પ્રદર્શનથી શ્રેષ્ઠ પ્રદર્શન કર્યું હોય.
 Log-prob (Fast λ): -558.8171915638965
 Log-prob (Grid λ): -570.0029685672033

Sentence: પાકિસ્તાનમાં ગધેડાની વસ્તી નોંધપાત્ર છે, જે તેને ચીન માટે મુખ્ય ભાગીદાર બનાવે છે.
 Log-prob (Fast λ): -178.6523324878704
 Log-prob (Grid λ): -181.41848994122788

Sentence: નાણાકીય લાભ વિશે ચિંતા કરશો નહીં, કારણ કે તે ભવિષ્યમાં તમારા માટે ખૂબ ફાયદાકારક રહેશે.
 Log-prob (Fast λ): -136.31303493922962
 Log-prob (Grid λ): -134.88906279309163

Sentence: મોટોરોલા વ્યવસાયમાં નુકસાન EBITDA માર્જિન પર વધુ દબાણ લાવી શકે છે.
 Log-prob (Fast λ): -135.40057136713943
 Log-prob (Grid λ): -137.24322635537854

Sentence: ટીમ ઈન્ડિયા ઓસ્ટ્રેલિયા સામે ત્રણ મેચની ODI શ્રેણી રમશે.
 Log-prob (Fast λ): -90